![Slide 1](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/1.svg)

![Slide 2](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/2.svg)

![Slide 3](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/3.svg)

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

![Slide 4](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/4.svg)

## Demo 1 — "The tool-call gap"

> *LLMs don't run tools — they request them.*

![Slide 5](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/5.svg)

![Slide 6](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/6.svg)

In [ ]:
# A tool call is just a data structure — no execution happens here
from llm_agents_from_scratch.data_structures.tool import ToolCall

tool_call = ToolCall(
    tool_name="hailstone_step",
    arguments={"x": 6},
)
tool_call

In [ ]:
# SimpleFunctionTool: turn any Python function into an LLM-compatible tool
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool


def hailstone_step(x: int) -> int:
    """Performs a single step of the Hailstone sequence."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


hailstone_tool = SimpleFunctionTool(hailstone_step)

print(f"Name:        {hailstone_tool.name}")
print(f"Description: {hailstone_tool.description}")
print(f"Schema:      {hailstone_tool.parameters_json_schema}")

In [ ]:
from llm_agents_from_scratch.llms.ollama import OllamaLLM

llm = OllamaLLM(model="qwen3:14b", think=False)

# The LLM sees the schema and decides to request a tool call
user_msg, response_msg = await llm.chat(
    "What is the next number in the Hailstone sequence after 6?",
    tools=[hailstone_tool],
)

# The LLM output — not a result, a *request*
print(response_msg.tool_calls)

In [ ]:
# We execute the tool — the LLM didn't
tool_call = response_msg.tool_calls[0]
tool_call_result = hailstone_tool(tool_call)
print(tool_call_result)

In [ ]:
# Close the loop: feed the result back — this is the gap made visible
_, final_response = await llm.continue_chat_with_tool_results(
    tool_call_results=[tool_call_result],
    chat_history=[user_msg, response_msg],
)
print(final_response)

## Demo 2 — "Closing the loop"

> *Wrapping the tool-call loop in `LLMAgent` + `TaskHandler` is what makes it an agent.*

![Slide 7](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/7.svg)

![Slide 8](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/8.svg)

In [ ]:
import logging

from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

In [ ]:
# Obfuscated tool: the agent must reason, not pattern-match the name
from pydantic import BaseModel

from llm_agents_from_scratch.tools import PydanticFunctionTool


class AlgoParams(BaseModel):
    """Params for next_number."""

    x: int


def next_number(params: AlgoParams) -> int:
    """Generate the next number of the sequence."""
    if params.x % 2 == 0:
        return params.x // 2
    return 3 * params.x + 1


tool = PydanticFunctionTool(next_number)

In [ ]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task

llm_agent = LLMAgent(llm=llm, tools=[tool])

instruction_template = """
You are given a tool, `next_number`, that generates the next number in the
sequence given the current number.

Start with the number x={x}.

<rules>
CALL `next_number` on the current number x
STOP AND WAIT for the result.
REPEAT this step-by-step process until the number 1 is reached.
FINAL RESULT: When you receive the number 1, provide the complete sequence you
observed from start to finish (including the starting number x and ending number 1).
</rules>

<warnings>
NEVER fabricate or simulate tool call results
NEVER make multiple tool calls in one response
STOP and WAIT - ALWAYS wait for the actual tool response before deciding next steps
</warnings>
""".strip()

task = Task(instruction=instruction_template.format(x=6))
handler = llm_agent.run(task, max_steps=15)

In [ ]:
print(f"Done:        {handler.done()}")
print(f"Steps taken: {handler.step_counter}")
print(f"\nResult: {handler.result()}")

In [ ]:
# The rollout is the agent's working memory — every step recorded
print(handler.rollout)

In [ ]:
import asyncio

# Same agent, two tasks, running concurrently — the surprise
task_a = Task(instruction=instruction_template.format(x=3))
task_b = Task(instruction=instruction_template.format(x=7))

handler_a = llm_agent.run(task_a, max_steps=20)
handler_b = llm_agent.run(task_b, max_steps=20)

await asyncio.gather(handler_a.background_task, handler_b.background_task)

print(
    f"Task A ({3}) — steps: {handler_a.step_counter}, result: {handler_a.result()}"
)
print(
    f"Task B ({7}) — steps: {handler_b.step_counter}, result: {handler_b.result()}"
)

## Demo 3 — "Tools as a protocol"

> *The entire tool ecosystem in one line.*

![Slide 9](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/9.svg)

![Slide 10](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/10.svg)

In [ ]:
# Real-world MCP: GitHub's server over streamable HTTP
# Pre-run this cell — live network dependency
import getpass
import os

os.environ["GITHUB_PAT"] = getpass.getpass("GitHub PAT: ")

github_mcp_provider = MCPToolProvider(
    name="github_mcp",
    streamable_http_url="https://api.githubcopilot.com/mcp/",
    streamable_http_headers={
        "Authorization": f"Bearer {os.environ['GITHUB_PAT']}",
    },
)

In [ ]:
from llm_agents_from_scratch import LLMAgentBuilder

github_agent = (
    await LLMAgentBuilder()
    .with_llm(llm)
    .with_mcp_provider(github_mcp_provider)
    .build()
)

for t in github_agent.tools:
    print(f"{t.name}: {(t.description or '')[:60]}...")

In [ ]:
# Self-referential: the agent fetches the latest release of this very repo
task = Task(
    instruction=(
        "Use the get_latest_release tool to get the latest release for the "
        "GitHub repo with owner 'nerdai' and repo 'llm-agents-from-scratch'."
    ),
)
handler = github_agent.run(task, max_steps=10)
result = handler.exception() or handler.result()
print(result)

## Demo 4 — "Giving the agent a playbook"

> *Not what it can do — how it thinks.*

![Slide 11](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/11.svg)

![Slide 12](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/12.svg)

In [ ]:
from llm_agents_from_scratch.data_structures.skill import SkillScope
from llm_agents_from_scratch.skills.discovery import discover_skills

skills = discover_skills([SkillScope.PROJECT])
print(f"Discovered {len(skills)} skill(s):")
for name, skill in skills.items():
    print(f"  {name}: {skill.frontmatter.description}")

In [ ]:
# Pause here — this is the agent's instruction set, written in plain text
skill = skills["hailstone-sequence"]
print(skill.read_body())

In [ ]:
# Activate the skill — the programmatic slash command
handler = llm_agent.run_with_skill(
    skill_name="hailstone-sequence",
    prompt="Compute the Hailstone sequence starting from 12.",
    max_steps=25,
)
result = handler.exception() or handler.result()
print(f"Steps taken: {handler.step_counter}")
print(f"\nResult:\n{result}")

## Demo 5 — "Giving the agent a past"

> *The rollout is working memory for one task. What about memory across tasks?*

![Slide 13](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/13.svg)

In [ ]:
# reflective_memory: after each task the LLM distils a one-sentence lesson
# FastEmbed model is pre-cached — no download delay during the demo
from llm_agents_from_scratch.memory.recipes import reflective_memory

memory = reflective_memory(llm=llm, max_results=1)
agent_with_memory = LLMAgent(llm=llm, tools=[tool], memories=[memory])

In [ ]:
# First run — no prior experience
task = Task(instruction=instruction_template.format(x=6))
handler = agent_with_memory.run(task, max_steps=15)
print(handler.result())

In [ ]:
# The money shot — episode stored with the reflection
episodes = await memory.store.read_recent(1)
print(str(episodes[0]))

In [ ]:
# Second run — agent starts with the lesson from Run 1 in its system prompt
task2 = Task(instruction=instruction_template.format(x=6))
handler2 = agent_with_memory.run(task2, max_steps=15)
print(handler2.result())

![Slide 14](https://d3ddy8balm3goa.cloudfront.net/talks/2026/acm-tech-talks/14.svg)